# Integrated Gradients

In [ ]:
# loading in the model
import torch
import os
import random
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from PIL import Image

import torch.nn.functional as F
import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             ConfusionMatrixDisplay, RocCurveDisplay)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = resnet18(weights=None)  # avoids any download
model.fc = nn.Linear(model.fc.in_features, 2)

state = torch.load("best_model.pth", map_location=DEVICE)
model.load_state_dict(state)
model.to(DEVICE).eval()

### Loading in the data
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

BATCH_SIZE = 32  # keep consistent


def load_npz_split(split):
    data = np.load(f"data/preprocessed/{split}.npz")
    images = torch.from_numpy(data["images"]).float()
    labels = torch.from_numpy(data["labels"]).long()
    return TensorDataset(images, labels)


train_dataset = load_npz_split("train")
val_dataset = load_npz_split("val")
test_dataset = load_npz_split("test")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

## Baseline choice

Inputs are ImageNet-normalized, so a zero tensor in the model's input space is **not** a black image — it corresponds to the ImageNet mean colour (mid-grey). For chest X-rays, the more principled baseline is a **true black image** (pixel value 0), which after normalization becomes `(0 − mean) / std`. In X-ray physics, black ≈ no attenuation ≈ air, i.e. genuine absence of signal, so attributions read as "what does the model gain over a blank exposure". Alternatives are Gaussian noise (averaged across draws) or a blurred input; we use the black-image baseline as default.

In [ ]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)


def black_baseline_like(x):
    """Black pixel image (zeros in raw pixel space) mapped to normalized space."""
    return (torch.zeros_like(x) - IMAGENET_MEAN) / IMAGENET_STD


@torch.no_grad()
def denormalize(x):
    return (x * IMAGENET_STD + IMAGENET_MEAN).clamp(0, 1)


def integrated_gradients(model, x, target, baseline=None, steps=16):
    """Riemann-midpoint Integrated Gradients for a single image.

    Args:
        x:        (1, C, H, W) input in normalized space
        target:   target class index (int)
        baseline: (1, C, H, W) tensor; defaults to black-image baseline
        steps:    number of Riemann steps (default 16)

    Returns:
        attribution: (1, C, H, W) tensor on CPU
        delta:       completeness gap |sum(IG) − (f(x) − f(baseline))|
    """
    model.eval()
    x = x.to(DEVICE)
    if baseline is None:
        baseline = black_baseline_like(x)
    baseline = baseline.to(DEVICE)

    # Midpoint rule: alphas = (k + 0.5) / steps,  k = 0..steps-1
    alphas = (torch.arange(steps, device=DEVICE).float() + 0.5) / steps
    interp = baseline + alphas.view(-1, 1, 1, 1) * (x - baseline)   # (S, C, H, W)
    interp.requires_grad_(True)

    logits = model(interp)
    score = logits[:, target].sum()                                 # sum → per-sample grad
    grads = torch.autograd.grad(score, interp)[0]                   # (S, C, H, W)

    attribution = (x - baseline) * grads.mean(dim=0, keepdim=True)

    with torch.no_grad():
        f_x = model(x)[0, target]
        f_b = model(baseline)[0, target]
        delta = abs(attribution.sum().item() - (f_x - f_b).item())

    return attribution.detach().cpu(), delta

In [ ]:
CLASSES = ["NORMAL", "PNEUMONIA"]


def show_ig(image, attribution, label_idx, pred_idx, title=""):
    img  = denormalize(image.to(DEVICE))[0].cpu().permute(1, 2, 0).numpy()
    attr = attribution[0].sum(dim=0).numpy()                    # collapse channels
    vmax = np.percentile(np.abs(attr), 99) + 1e-12

    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    axes[0].imshow(img)
    axes[0].set_title(f"input  (label={CLASSES[label_idx]}, pred={CLASSES[pred_idx]})")
    axes[1].imshow(attr, cmap="seismic", vmin=-vmax, vmax=vmax)
    axes[1].set_title("IG attribution (signed, summed over channels)")
    axes[2].imshow(img)
    axes[2].imshow(attr, cmap="seismic", vmin=-vmax, vmax=vmax, alpha=0.5)
    axes[2].set_title("overlay")
    for ax in axes: ax.axis("off")
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


# Run IG on one example per class from the test set, target = predicted class
shown = {0: False, 1: False}
for img, lbl in test_dataset:
    lbl_i = int(lbl)
    if shown[lbl_i]:
        continue
    shown[lbl_i] = True

    x = img.unsqueeze(0)
    with torch.no_grad():
        pred_i = int(model(x.to(DEVICE)).argmax(dim=1))

    attr, delta = integrated_gradients(model, x, target=pred_i, steps=16)
    show_ig(
        x, attr, lbl_i, pred_i,
        title=f"IG · black baseline · 16 Riemann steps · completeness Δ={delta:.3e}",
    )

    if all(shown.values()):
        break

In [ ]:
# 5 NORMAL + 5 PNEUMONIA test samples — IG vs black baseline, 16 steps
N_PER_CLASS = 5
samples = {0: [], 1: []}
for img, lbl in test_dataset:
    li = int(lbl)
    if len(samples[li]) < N_PER_CLASS:
        samples[li].append(img)
    if all(len(v) >= N_PER_CLASS for v in samples.values()):
        break


def plot_ig_grid(images, title):
    n = len(images)
    fig, axes = plt.subplots(n, 3, figsize=(11, 3.0 * n))
    for i, img in enumerate(images):
        x = img.unsqueeze(0)
        with torch.no_grad():
            pred_i = int(model(x.to(DEVICE)).argmax(dim=1))
        attr, delta = integrated_gradients(model, x, target=pred_i, steps=16)

        rgb  = denormalize(x.to(DEVICE))[0].cpu().permute(1, 2, 0).numpy()
        a    = attr[0].sum(dim=0).numpy()
        vmax = np.percentile(np.abs(a), 99) + 1e-12

        axes[i, 0].imshow(rgb)
        axes[i, 0].set_title(f"input · pred={CLASSES[pred_i]}")
        axes[i, 1].imshow(a, cmap="seismic", vmin=-vmax, vmax=vmax)
        axes[i, 1].set_title(f"IG (Δ={delta:.2e})")
        axes[i, 2].imshow(rgb)
        axes[i, 2].imshow(a, cmap="seismic", vmin=-vmax, vmax=vmax, alpha=0.5)
        axes[i, 2].set_title("overlay")
        for ax in axes[i]:
            ax.axis("off")
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()


plot_ig_grid(samples[0], "IG · 5 NORMAL test samples · black baseline · 16 steps")
plot_ig_grid(samples[1], "IG · 5 PNEUMONIA test samples · black baseline · 16 steps")

In [ ]:
# Baseline sensitivity — same image, four different baselines
def make_baselines(x):
    """Return dict {name: baseline_tensor} all in normalized space."""
    x = x.to(DEVICE)
    # 1) Black image (zero pixels) → -mean/std
    b_black = black_baseline_like(x)
    # 2) Zero tensor in normalized space (= ImageNet mean grey in pixel space)
    b_zero = torch.zeros_like(x)
    # 3) Gaussian noise (independent draw, fixed seed for reproducibility)
    g = torch.Generator(device=DEVICE).manual_seed(0)
    b_noise = torch.randn(x.shape, generator=g, device=DEVICE)
    # 4) Heavily blurred input — preserves low-frequency content only
    k, c = 51, x.shape[1]
    pad = k // 2
    kernel = torch.ones(c, 1, k, k, device=DEVICE) / (k * k)
    b_blur = F.conv2d(F.pad(x, (pad,) * 4, mode="reflect"), kernel, groups=c)
    return {
        "black (zero pixels)":          b_black,
        "zero tensor (= ImageNet grey)": b_zero,
        "Gaussian noise":                b_noise,
        "blurred input":                 b_blur,
    }


# Pick first PNEUMONIA test image
for img, lbl in test_dataset:
    if int(lbl) == 1:
        x_demo = img.unsqueeze(0)
        break
with torch.no_grad():
    pred_demo = int(model(x_demo.to(DEVICE)).argmax(dim=1))

baselines = make_baselines(x_demo)
results = {
    name: integrated_gradients(model, x_demo, target=pred_demo, baseline=b, steps=16)
    for name, b in baselines.items()
}

rgb = denormalize(x_demo.to(DEVICE))[0].cpu().permute(1, 2, 0).numpy()
fig, axes = plt.subplots(len(results), 3, figsize=(11, 3.0 * len(results)))
for i, (name, (attr, delta)) in enumerate(results.items()):
    b_rgb = denormalize(baselines[name])[0].cpu().permute(1, 2, 0).numpy()
    a     = attr[0].sum(dim=0).numpy()
    vmax  = np.percentile(np.abs(a), 99) + 1e-12

    axes[i, 0].imshow(b_rgb)
    axes[i, 0].set_title(f"baseline: {name}")
    axes[i, 1].imshow(a, cmap="seismic", vmin=-vmax, vmax=vmax)
    axes[i, 1].set_title(f"IG (Δ={delta:.2e})")
    axes[i, 2].imshow(rgb)
    axes[i, 2].imshow(a, cmap="seismic", vmin=-vmax, vmax=vmax, alpha=0.5)
    axes[i, 2].set_title("overlay on input")
    for ax in axes[i]:
        ax.axis("off")

plt.suptitle(f"Baseline sensitivity · same PNEUMONIA image · target={CLASSES[pred_demo]} · 16 steps",
             fontsize=13)
plt.tight_layout()
plt.show()

## Discussion — Q3.2 / Q3.3 / Q3.4

The answers below are **hypotheses to verify against the figures above**. Look at the 5+5 grid for Q3.2/Q3.3 and at the baseline-sensitivity figure for Q3.4.

### Q3.2 — Do the maps highlight sensible regions?

**What "sensible" means here:** positive (red) attribution should sit *inside the chest cavity*, ideally on the lung fields where opacities live for PNEUMONIA, and on thoracic structures (heart border, mediastinum, diaphragm) more generally. Negative (blue) attribution typically lands on tissue that argues *against* the predicted class — for PNEUMONIA, often the clear lung areas.

**Red flags to watch for** (these would mean the model is using shortcuts, not radiology):
- attribution concentrated in image corners, padding, or text/markers burned into the X-ray
- attribution riding along the dark background outside the ribcage
- attribution on scanner artefacts or rotation/aspect-ratio cues

**Initial read (to confirm against the figure):** maps should land *inside the chest cavity*, with PNEUMONIA cases showing dense, diffuse attribution across the lung fields where consolidations sit, and NORMAL cases showing sparser, less localized attribution. → **likely sensible at the type-of-region level**, but inspect corners and burned-in markers before declaring victory.

### Q3.3 — Are attributions consistent across samples?

Two distinct meanings — and they don't always agree:

1. **Within-class type-of-region consistency** — do all PNEUMONIA maps fire on lung tissue / opacities, even though the *exact spatial location* differs from case to case? This is what we *want* to see: real disease is heterogeneous, so attribution should follow the pathology, not a fixed pixel pattern.
2. **Spatial pixel-level consistency** — does attribution always light up the *same coordinates*? This would actually be a **bad sign**: it would suggest a positional shortcut (e.g. "the bottom-left of the image is always pneumonia"), not a learned disease pattern.

**Initial read:** expect *type-of-region* consistency (lung fields for PNEUMONIA) with spatial variability case-to-case. NORMAL maps tend to be sparser and noisier — there is no canonical "absence" pattern, so an asymmetry between the two classes is itself expected and not a failure mode.

### Q3.4 — Does baseline choice matter?

**Yes, structurally.** IG is baseline-relative by construction: the formula `(x − b) · ∫∂f/∂x dα` answers the question *"what makes x more like the predicted class than b does"*. Change `b`, change the question.

What to look for in the baseline-sensitivity figure:
- **Black** baseline → attributes heavily to *bright* structures (lungs, mediastinum), since every bright pixel is "far from black".
- **Zero tensor (ImageNet grey)** → weaker / shifted attribution, because mid-grey is closer to many lung pixels in normalized space, so the `(x − b)` factor is smaller there.
- **Gaussian noise** → noisier maps, but cancels positional priors that a fixed baseline can introduce; for stable maps you'd average over several noise draws (Expected Gradients).
- **Blurred input** → attribution localizes to *high-frequency* structure (edges, ribs, opacity boundaries) only, since low-frequency content is shared with the baseline and cancels.

**Initial read:** the *qualitative pattern* (chest interior, lung fields) is roughly stable across baselines, but the *intensity distribution* and *which structures dominate* shift visibly. Treat any single IG map as one slice through baseline-space — your hypothesis that maps are sensible AND consistent AND baseline-sensitive is consistent with what IG theory predicts.